# DeepCGM Tutorial - Knowledge-Guided Deep Learning Crop Growth Model

> Hands-on tutorial for the master-level course on **data-driven crop modelling**.
> Case study: **DeepCGM** (Han et al., 2025, *Field Crops Research*).
>
> Tutorial repository: https://github.com/flydephone/DeepCGM_tutorial
> Upstream paper repository: https://github.com/WUR-AI/DeepCGM

---

## Learning objectives

After completing this notebook you will be able to:

1. Explain why a pure data-driven LSTM struggles when crop observations are sparse, and why a process-driven model alone is also limited.
2. Describe the three layers of domain knowledge DeepCGM injects: **mass conservation**, **Input Mask** (relevant-inputs-only), **Convergence Loss** (stable internal processes).
3. Observe and analyse the four experiments on slide 6 of the course slides, all using the **700-epoch pretrained weights** released by the authors:
   - Task 1 - NaiveLSTM on sparse observations;
   - Task 2.1 - DeepCGM with mass conservation only;
   - Task 2.2 - DeepCGM with Input Mask added;
   - Task 2.3 - DeepCGM with Convergence Loss added (full configuration).
4. Compare every configuration across the six observed variables (PAI / WLV / WST / WSO / WAGT / Yield).
5. Watch the training evolution of LSTM vs DeepCGM+IM+CG side by side as a GIF.
6. Answer the discussion questions, and optionally tackle the training-based exercises at the end.

> The required exercises are observation/analysis only - no training is necessary. The bonus training-evolution section is the only place a model is trained from scratch.

## 0.1 Background - why Knowledge-Guided Machine Learning?

| Approach | Strengths | Limitations |
| --- | --- | --- |
| Process-based models (ORYZA2000, WOFOST) | Interpretable, physiologically grounded | Hard to calibrate, oversimplified |
| Pure data-driven (LSTM) | Powerful fitter | Black box, data-hungry, prone to over-fitting |
| **Knowledge-guided DL (DeepCGM)** | Learns physically plausible trajectories even on sparse data | Higher design cost, requires domain knowledge |

The core idea of DeepCGM: **let data adapt the model while domain knowledge keeps it physically plausible**. This is implemented as three knowledge layers:

1. **Mass conservation** - based on MC-LSTM (Hoedt et al., 2021). Cell states represent organ-level carbon pools whose transfers must conserve mass.
2. **Physiological meaning** - explicit ordering of photosynthesis -> maintenance respiration -> partition -> growth respiration -> inter-organ redistribution.
3. **Input Mask + Convergence Loss** - each sub-process only sees the inputs it physically depends on, and unobserved transitions are penalised when they jump unrealistically.

In the rest of this notebook we start from a pure LSTM baseline and add these layers one by one, observing how predictions improve.

## 0.2 How this notebook is organised

- **Part 1** loads the data, splits it into train/test, and loads the six pretrained 700-epoch configurations into memory.
- **Parts 2-5** look at Task 1 / 2.1 / 2.2 / 2.3 in turn - **no training needed**, just observation.
- **Part 6** does a side-by-side comparison of all six configurations.
- **Part 7** is a bonus: retrains LSTM and DeepCGM+IM+CG from scratch and animates the training evolution as a GIF (~15 min on CPU).
- **Parts 8-9** are discussion questions and optional training-based exercises.

Almost all the heavy code lives in [`helper.py`](https://github.com/flydephone/DeepCGM_tutorial/blob/main/helper.py). The notebook itself is intentionally short so you can focus on **what** each step does and **why** it matters, rather than on plotting code.

## 0.3 Setup - one-shot bootstrap

The cell below makes the notebook **self-contained**: if the upstream DeepCGM code (models, formatted data, pretrained weights) is not already present, it clones https://github.com/WUR-AI/DeepCGM into the current directory. If `helper.py` is not present, it downloads it from the tutorial repository.

> Works equally well in **Google Colab**, on **your own machine**, or inside an existing clone of the upstream DeepCGM repo.

In [ ]:
import os, urllib.request

UPSTREAM_GIT = "https://github.com/WUR-AI/DeepCGM.git"
HELPER_URL   = "https://raw.githubusercontent.com/flydephone/DeepCGM_tutorial/main/helper.py"

# 1) Ensure we have the upstream DeepCGM repository on disk
if not os.path.isdir('models_aux'):
    if not os.path.isdir('DeepCGM'):
        print("Cloning the upstream DeepCGM repository ...")
        ret = os.system(f"git clone --depth 1 {UPSTREAM_GIT}")
        assert ret == 0, "git clone failed - is git installed and do you have internet access?"
    os.chdir('DeepCGM')
print("Working directory:", os.getcwd())

# 2) Ensure helper.py is present
if not os.path.exists('helper.py'):
    print("Downloading helper.py ...")
    urllib.request.urlretrieve(HELPER_URL, 'helper.py')
print("helper.py ready (size:", os.path.getsize('helper.py'), "bytes)")

## 0.4 Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

import utils       # from upstream DeepCGM
import helper      # tutorial-specific helpers

print(f"PyTorch {torch.__version__}  |  device: {helper.device}")

---
# Part 1 - Data and pretrained models

## 1.1 Hyperparameters
These are the only knobs you may want to tweak. The defaults reproduce the paper's main setup.

In [ ]:
SEED         = 0       # which of the 50 robust runs to load (each is a different random seed)
TRA_YEAR     = "2018"  # "2018" or "2019"
BATCH_SIZE   = 128

helper.setup_seed(SEED)

## 1.2 Load the formatted rice-field dataset

In [ ]:
rea_ory, rea_par, rea_wea_fer, rea_spa, rea_int, max_min = helper.load_dataset(TRA_YEAR)
print(f"{len(rea_ory)} growing seasons available for year {TRA_YEAR}")

## 1.3 Look at one growing season

Pick any sample and visualise the meteorology (top row) and the sparse field observations (bottom row).

- Black dots = the sparse, real-world observations the model has to fit.
- Grey lines = the ORYZA2000 process-based simulation.

Notice how few observations there are in a typical growing season - this is exactly why injecting domain knowledge matters.

In [ ]:
helper.show_sample_overview(sample_idx=10,
                            rea_ory=rea_ory, rea_spa=rea_spa,
                            rea_wea_fer=rea_wea_fer, max_min=max_min)

## 1.4 Train/test split and pretrained models

The split follows the paper exactly: when training on 2018, the first 65 samples are training and the rest are testing. We then load all six 700-epoch pretrained configurations and cache their test-set predictions.

In [ ]:
tra_loader, tes_loader, n_tra, n_tes = helper.split_and_load(
    rea_ory, rea_wea_fer, rea_spa, rea_int, tra_year=TRA_YEAR, batch_size=BATCH_SIZE
)
print(f"Train samples: {n_tra:3d}  |  test samples: {n_tes:3d}")

pretrained_predictions = helper.load_all_pretrained(
    tes_loader, max_min, tra_year=TRA_YEAR, seed=SEED
)

---
# Part 2 - Task 1: NaiveLSTM on sparse observations

`NaiveLSTM` ([source](https://github.com/WUR-AI/DeepCGM/blob/main/models_aux/NaiveLSTM.py)) is the textbook baseline: a single-layer LSTM with hidden size 64 that maps the daily meteorological inputs to the six target variables. No mass conservation, no physiological structure - just gradient descent on the sparse observations.

Below we plot the 700-epoch pretrained NaiveLSTM on the last test sample.

In [ ]:
helper.show_task_result("LSTM", pretrained_predictions,
                        title="Task 1: NaiveLSTM (700-epoch pretrained)")

**What to look for**

- The red model curve passes close to the black observation dots on the days where observations exist.
- But on the unobserved days the curve typically wiggles unrealistically (e.g. WLV / WST going up and down before any leaves should exist).

This jagged behaviour on unobserved days is the symptom that motivates DeepCGM.

---
# Part 3 - Task 2.1: DeepCGM with mass conservation only

## 3.1 Model structure

`DeepCGM` ([source](https://github.com/WUR-AI/DeepCGM/blob/main/models_aux/DeepCGM_fast.py)) replaces the LSTM black box with five physiology-shaped gates:

| Gate | Shape | Meaning |
| --- | --- | --- |
| `C_assimilate_gate`     | (1, 1)    | Photosynthesis: converts potential C to actual assimilated C |
| `C_mainResp_gate`       | (1, 24)   | Maintenance respiration: loss ratio per carbon pool |
| `C_partitation_gate`    | (1, 24)   | Partitioning: distributes net C among 24 organ-cells |
| `C_growResp_gate`       | (1, 24)   | Growth respiration: additional cost during growth |
| `C_redistribution_gate` | (24, 24)  | Inter-organ carbon redistribution (leaf->stem->grain etc.) |

The cell below shows the actual one-step code in `rate()` together with links to the two architecture diagrams in the upstream repo.

In [ ]:
helper.show_deepcgm_architecture()

## 3.2 Results - DeepCGM base (no IM, no CG)

In [ ]:
helper.show_task_result("DeepCGM", pretrained_predictions,
                        title="Task 2.1: DeepCGM (base, no IM / no CG, 700-epoch pretrained)")

**Observation** - compared to the LSTM red line in Part 2, the DeepCGM red curve is already much more *crop-like*: no spurious oscillations and the carbon never appears or disappears without going through a modelled process. But because there is no constraint on *which inputs* the redistribution gate may use, the trajectory can still drift mid-season.

---
# Part 4 - Task 2.2: add the Input Mask

## 4.1 What is the Input Mask?

Quoting [DeepCGM_fast.py:111-117](https://github.com/WUR-AI/DeepCGM/blob/main/models_aux/DeepCGM_fast.py#L111-L117):

```python
self.C_i_redistribution_prior = Variable(torch.ones(self.input_num, ...))
if input_mask:
    self.C_i_redistribution_prior[self.C_cell_num + 1:] = 0
```

`input_num = 24 (cell) + 5 (aux) = 29`. With `input_mask` enabled, rows for `Rad / Tmax / Tmin / N_cum` are zeroed out, leaving only the 24 cell states and DVS. **Physical reading**: inter-organ carbon redistribution should depend on each organ's current carbon and the developmental stage, but not on today's weather or fertiliser. This is the *relevant-inputs-only* principle from the paper.

## 4.2 Results - DeepCGM + Input Mask

In [ ]:
helper.show_task_result("DeepCGM+IM", pretrained_predictions,
                        title="Task 2.2: DeepCGM + Input Mask (700-epoch pretrained)")

**Observation** - the trajectory becomes notably smoother because the redistribution gate can no longer be biased by daily weather noise. RMSE on WLV / WST / WAGT typically drops vs Task 2.1.

---
# Part 5 - Task 2.3: add the Convergence Loss (full DeepCGM)

## 5.1 What is the Convergence Loss?

From [train.py:43-51](https://github.com/WUR-AI/DeepCGM/blob/main/train.py#L43-L51):

```python
def CG_LOSS(pred, mask, aux_all, X):
    C_cell_all, C_cell_convergence_all, num_segment = aux_all
    ...
    C_converge_loss = mse_loss(C_cell_convergence_all, C_cell_all)
                        .masked_select(...).mean() * 1e5
```

`C_cell_all` is the cell state from normal forward iteration. `C_cell_convergence_all` is the redistributed state computed independently at every time step. Penalising their difference keeps adjacent time steps from jumping unrealistically when no observation is anchoring them - this is the *stable internal processes* principle.

## 5.2 Results - DeepCGM + IM + CG (the full DeepCGM)

In [ ]:
helper.show_task_result("DeepCGM+IM+CG", pretrained_predictions,
                        title="Task 2.3: DeepCGM + IM + CG (full, 700-epoch pretrained)")

**Observation** - this is the configuration the paper reports. The curves are smooth, monotonic where physiology requires it, and match both ORYZA and the sparse observations even on days with no anchor point.

---
# Part 6 - Side-by-side comparison

The figure below puts the six configurations next to each other on the same test sample. Rows are variables (PAI, WLV, WST, WSO, WAGT, Yield), columns are configurations.

In [ ]:
helper.compare_models(pretrained_predictions)

In [ ]:
df_rmse = helper.rmse_table(pretrained_predictions)

**Conclusions (matches Fig.5 / Fig.12 of the paper):**

1. LSTM can hit the sparse observation points but its in-between trajectory is often unphysical.
2. MC-LSTM (mass conservation alone) improves smoothness but still misses physiological constraints.
3. DeepCGM base (mass conservation + explicit photosynthesis / respiration / partition) further improves.
4. Adding **Input Mask + Convergence Loss** together gives the best average RMSE across variables.
5. The variables that benefit the most are **WLV / WST / WAGT**, exactly the ones with the sparsest mid-season observations.

---
# Part 7 - Bonus: training evolution animation (LSTM vs DeepCGM+IM+CG)

The released `model_weight/` directory only ships the final checkpoint of each robust run, so the 700-epoch trajectory itself is not available on disk. This bonus section **retrains** the two endpoints (LSTM and full DeepCGM+IM+CG) from scratch, recording the test-set predictions every `GIF_EPOCH_STEP` epochs, and assembles a side-by-side animation.

> Total cost on CPU: about **12-18 minutes** (mostly DeepCGM+IM+CG). Reduce `GIF_TOTAL_EPOCH` for a quicker preview.

In [ ]:
GIF_TOTAL_EPOCH = 700     # full paper setup
GIF_EPOCH_STEP  = 10      # 71 snapshots per model
GIF_FPS         = 5       # playback frame rate
GIF_SAMPLE_LOC  = -1      # test-set sample shown in the animation

In [ ]:
from models_aux.NaiveLSTM    import NaiveLSTM
from models_aux.DeepCGM_fast import DeepCGM

lstm_snaps = helper.train_with_snapshots(
    model_cls=NaiveLSTM, input_mask=False, lr=0.005, convergence_loss=False, tag="LSTM",
    total_epochs=GIF_TOTAL_EPOCH, snap_every=GIF_EPOCH_STEP,
    tra_loader=tra_loader, tes_loader=tes_loader, max_min=max_min, seed=SEED,
)
dcgm_snaps = helper.train_with_snapshots(
    model_cls=DeepCGM, input_mask=True, lr=0.1, convergence_loss=True, tag="DeepCGM+IM+CG",
    total_epochs=GIF_TOTAL_EPOCH, snap_every=GIF_EPOCH_STEP,
    tra_loader=tra_loader, tes_loader=tes_loader, max_min=max_min, seed=SEED,
)
print(f"LSTM snapshots: {len(lstm_snaps)}  |  DeepCGM+IM+CG snapshots: {len(dcgm_snaps)}")

In [ ]:
n_frames = helper.make_evolution_gif(
    lstm_snaps, dcgm_snaps,
    gif_path='figure/training_evolution.gif',
    fps=GIF_FPS, sample_loc=GIF_SAMPLE_LOC, total_epochs=GIF_TOTAL_EPOCH,
)
print(f"GIF written ({n_frames} frames @ {GIF_FPS} fps)")

from IPython.display import Image
Image('figure/training_evolution.gif')

**What to watch in the animation**

- **Early (epoch 0-50)**: both rows are still mostly noise.
- **Middle (epoch 50-200)**: LSTM has already pinned the observation points but keeps oscillating between them; DeepCGM gradually grows physically plausible S-curves.
- **Late (epoch 200-700)**: LSTM's oscillations barely improve - more epochs cannot fix the lack of physical constraints. DeepCGM keeps smoothing and tracks ORYZA closely.

This is the most direct demonstration of *why knowledge-guided learning beats simply training longer on sparse data*.

---
# Part 8 - Discussion questions (must-do)

Answer in short paragraphs - no additional experiments are needed.

1. **Mass conservation vs the extra constraints.** Looking at Part 6, the DeepCGM base already beats LSTM / MC-LSTM. What did Input Mask and Convergence Loss add on top that the structural mass conservation alone could not solve?
2. **Observation density.** This dataset has ~5-6 observations per growing season. Suppose we had 2 observations per week. How would the relative benefit of the three knowledge layers change? Which one might become "nice to have" rather than essential?
3. **DVS comes from ORYZA.** The model receives DVS from `ORY[:, :, 0]` (i.e. the ORYZA simulation), not learned from data. What would likely break if you removed that dependency and forced DeepCGM to learn DVS internally?
4. **Cross-crop transfer.** To apply DeepCGM to maize or wheat, which components would need changing? (Hint: organ types, `C_cell_num`, and the hard-coded conversion constants `0.419 / 0.431 / 0.487`.)

---
# Part 9 - Optional exercises (training-based, placeholder)

> The detailed prompts will be added once the course staff has finalised them. The training utilities you will need are already exposed via `helper.*`:
>
> - `helper.train_loop(...)` - generic Adam training loop;
> - `helper.run_one_epoch(...)` - one pass of forward + (optional) backward;
> - `helper.FITTING_LOSS`, `helper.CG_LOSS` - loss building blocks;
> - `helper.train_with_snapshots(...)` - useful if you want to animate your own variant.
>
> Suggested directions (to be confirmed):
> 1. **Custom Input Mask** - change which sub-process the mask is applied to (e.g. partition instead of redistribution) and compare WLV / WST / WSO RMSE.
> 2. **Convergence-loss weight** - replace the `1e5` factor in `CG_LOSS` with `1e3` or `1e7` and analyse the bias/variance trade-off.
> 3. **Cross-year generalisation** - train on 2018, evaluate on 2019, and check which knowledge layer transfers best.
> 4. **Seed robustness** - re-run DeepCGM+IM+CG with SEED in {0,1,2,3,4} and plot a per-variable RMSE box-plot.
> 5. **Decision interpretation** - extract `C_partitation_mat` from a trained DeepCGM and heat-map it before vs after heading. Explain why C should flow to grain post-heading.

---
## References

- Han et al. (2025). *Knowledge-guided machine learning with multivariate sparse data for crop growth modelling*. Field Crops Research. [DOI 10.1016/j.fcr.2025.109885](https://doi.org/10.1016/j.fcr.2025.109885)
- Hoedt et al. (2021). *MC-LSTM: Mass-Conserving LSTM*. ICML.
- Upstream code: https://github.com/WUR-AI/DeepCGM
- This tutorial: https://github.com/flydephone/DeepCGM_tutorial